In [7]:
# ! pip install brotli

In [8]:
### This panel goes as a sperate file scrape_nse_new.py  ###
#########################################################################################
# NSE Stock/Indices Scrapper
# Source: https://github.com/alloc7260/NSE
#########################################################################################
import math
import requests
import datetime
import json
import urllib
import pandas as pd
import concurrent
from concurrent.futures import ALL_COMPLETED
# import logging
import brotli
import nselib
from nselib import capital_market 
#from datetime import datetime, timedelta



def fetch_quote(sname, start_date, end_date):
    # print("fetching - " + sname + " from " + start_date + " to " + end_date)
    # df = capital_market.price_volume_data(symbol=sname, from_date=start_date, to_date=end_date )
    df = capital_market.price_volume_data(symbol=sname, period="1Y" )
    # print("fetched - ")
    # print(df.head())
    return df

def scrape_data(start_date, end_date, name=None, input_type='stock'):

    start_date = datetime.datetime.strptime(start_date, "%d-%m-%Y")
    end_date = datetime.datetime.strptime(end_date, "%d-%m-%Y")

    # parts = name.split(sep="-")
    # sname = parts[0]
    # result = fetch_quote(name, start_date.strftime('%d-%m-%Y'), end_date.strftime('%d-%m-%Y'))
    result = fetch_quote(name, start_date.strftime('%d-%m-%Y'), end_date.strftime('%d-%m-%Y'))    
    
    # with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    #     future_to_sname = {executor.submit(fetch_quote, sname=sname
    #                                        ,start_date = start_date.strftime('%d-%m-%Y')
    #                                        , end_date=end_date.strftime('%d-%m-%Y')): sname for sname in [sname]}
    #     concurrent.futures.wait(future_to_sname, return_when=ALL_COMPLETED)
    #     for future in concurrent.futures.as_completed(future_to_sname):
    #         url = future_to_sname[future]
    #         try:
    #             df = future.result()
    #             result = df
    #         except Exception as exc:
    #             # logging.error('%r generated an exception: %s. Please try again later.' % (url, exc))
    #             raise exc
    return format_dataframe_result(result, start_date, end_date)


def format_dataframe_result(result, start_date, end_date):
    if result.empty:
        return f"No Data Found : for date range {start_date} to {end_date}"

    
    result.columns = ['symbol','Series', 'date','PrevClose','open','high','low','LastPrice','close','AveragePrice','volume','TurnOver','No.Trades']
    result.replace(',', '', regex=True, inplace=True)
    result.astype({'PrevClose': 'float', 'open': 'float', 'high': 'float', 'low': 'float', 'LastPrice': 'float', 'close': 'float', 'AveragePrice': 'float', 'volume': 'int', 'TurnOver': 'float', 'No.Trades': 'int'})
    # result['52WH'] = result['close'].max()
    # result['52WL'] = result['close'].min()    
    # result.astype({'52WH': 'float', '52WL': 'float'})
    result['date'] = pd.to_datetime(result['date'])
    result = result.sort_values('date', ascending=True)
    result.reset_index(drop=True, inplace=True)
    return result

def Get_Stock_Data(_ticker, _number_of_recent_days, _type="stock"):
    _number_of_weekends = (_number_of_recent_days / 5) * 2
    _number_of_trading_days =  _number_of_recent_days + _number_of_weekends + 30 # 30 is to cover for other holidays
    _start_date = datetime.datetime.today() - datetime.timedelta(_number_of_trading_days)
    _start_date_str = _start_date.strftime('%d-%m-%Y')
    _todays_date_str = datetime.datetime.today().strftime('%d-%m-%Y')
    _data_frame = scrape_data(_start_date_str, _todays_date_str, _ticker, _type)
    _data_frame = _data_frame.tail(_number_of_recent_days)
    return(_data_frame)

# def Get_Stock_Data2(_tickers, _number_of_recent_days, _type="stock"):

#     df1 = Get_Stock_Data(_tickers[0],_number_of_recent_days)
#     # df1 = df1.set_index("date")
    
#     for tkr in _tickers[1:]:
#         try:
#             t = Get_Stock_Data(tkr,_number_of_recent_days)
#             df1 = pd.concat([df1, t])
#         except:
#             print("error - "+ tkr)
#     return df1



def Get_Stock_Data3(_tickers, _number_of_recent_days, _type="stock", MAX_THREADS=5):
    print("Fetching data for " + str(len(_tickers)) + " tickers using " + str(MAX_THREADS) + " threads."    )
    # df1 = pd.DataFrame()

    result = pd.DataFrame()
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        future_to_Get_Stock_Data = {executor.submit(Get_Stock_Data, tkr, _number_of_recent_days): tkr for tkr in _tickers}
        concurrent.futures.wait(future_to_Get_Stock_Data, return_when=ALL_COMPLETED)
        for future in concurrent.futures.as_completed(future_to_Get_Stock_Data):
            tkr = future_to_Get_Stock_Data[future]
            try:
                df = future.result()
                result = pd.concat([result, df])
            except Exception as exc:
                # logging.error('%r generated an exception: %s. Please try again later.' % (url, exc))
                print("error - "+ tkr +" - " +  str(exc))
                # raise exc
    return result

# stklist = ['TVSMOTOR']
# Get_Stock_Data3(stklist, 500, MAX_THREADS=10)

In [9]:
# stklist = ['HDFCBANK','AMBUJACEM','ITC']
# Get_Stock_Data3(stklist, 500, MAX_THREADS=10)